# 'Gemini 3.8 TTS 가격 사용법' 글의 후속으로, 새 음성 리믹스 기능이 음성 디자인·복제와 어떻게 다른지와 저장한 음성을 API로 쓰는 법 정리

위에서부터 차례로 실행해 보세요.

### 준비: API 키 넣기

Colab 왼쪽의 🔑(보안 비밀)에 `GEMINI_API_KEY` 를 자기 값으로 추가하고 노트북 액세스를 켜 주세요. 키는 노트북에 저장되지 않아요.

In [ ]:
import os
from google.colab import userdata

for name in ['GEMINI_API_KEY']:
    os.environ[name] = userdata.get(name)

### SDK 설치와 클라이언트 확인

In [ ]:
%%bash
pip install -q -U "google-genai>=2.25.0" numpy scipy matplotlib
python -c "import google.genai as g;print(g.__version__)"

### 음성 디자인·저장

In [ ]:
%%bash
mkdir -p outputs
cat > d.py <<'E'
import os,base64
from google import genai
c=genai.Client(api_key=os.environ["GEMINI_API_KEY"],http_options={"api_version":"v1beta"})
v=c.voices.create(store=True,voice={"model":"gemini-3.8-flash-tts","type":"prompted","display_name":"Warm British Astronomer","gender":"male","language_code":"en-GB","prompted":{"input":"A warm, thoughtful astronomer in his late 60s with a gentle British accent, speaking with quiet wonder."}})
d=v.model_dump()
print({k:(x if k!='sample_audio' else 'b64') for k,x in d.items()})
open('outputs/design.wav','wb').write(base64.b64decode(d['sample_audio']['data']))
open('vid.txt','w').write(d['id'])
E
python d.py

### 리믹스 3회 호출

In [ ]:
%%bash
cat > r.py <<'E'
import os,base64,json
from google import genai
c=genai.Client(api_key=os.environ["GEMINI_API_KEY"],http_options={"api_version":"v1beta"})
BASE=open('vid.txt').read()
ids={}
for name,txt in [('remix1',"Add a subtle Southern US accent and speak a bit faster."),('remix2',"Add a subtle Southern US accent and speak a bit faster."),('remix3',"Make the voice higher-pitched and more energetic.")]:
    d=c.voices.create(store=True,voice={"type":"prompted","display_name":name,"gender":"male","language_code":"en-GB","prompted":{"base_voice":BASE,"input":txt}}).model_dump()
    open(f'outputs/{name}.wav','wb').write(base64.b64decode(d['sample_audio']['data']))
    ids[name]=d['id']; print(name,d['id'],d['prompted'])
json.dump(ids,open('ids.json','w'))
E
python r.py

### 저장한 음성 목록 조회

In [ ]:
%%bash
cat > l.py <<'E'
import os
from google import genai
c=genai.Client(api_key=os.environ["GEMINI_API_KEY"],http_options={"api_version":"v1beta"})
r=c.voices.list(type_=["prompted"],page_size=50)
for v in r.model_dump()['voices']: print(v['id'],v['type'],v['display_name'],(v.get('prompted') or {}).get('base_voice'))
E
python l.py

### voice ID로 반복 합성

In [ ]:
%%bash
cat > s.py <<'E'
import os,base64,json
from google import genai
c=genai.Client(api_key=os.environ["GEMINI_API_KEY"],http_options={"api_version":"v1beta"})
BASE=open('vid.txt').read(); R1=json.load(open('ids.json'))['remix1']
TEXT="The stars have been telling stories for billions of years, and tonight we finally have time to listen."
for vid,tag in [(BASE,'base'),(R1,'remix1')]:
    for i in range(3):
        it=c.interactions.create(model="gemini-3.8-flash-tts",input=[{"type":"user_input","content":[{"type":"text","text":TEXT}]}],response_format={"type":"audio"},generation_config={"speech_config":[{"voice":vid}]})
        open(f'outputs/synth_{tag}_{i+1}.wav','wb').write(base64.b64decode(it.model_dump()['output_audio']['data']))
E
python s.py && ls -la outputs